In [7]:
import warnings, time
from pathlib import Path
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option('display.max_columns', None)
plt.rcParams.update({'figure.dpi': 100})


In [8]:
# config
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data' / 'processed_data').exists() and PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

CONFIG = {
    'seed': 7524,
    'data_dir': PROJECT_ROOT / 'data' / 'processed_data',
    'text_col': 'message',
    'label_col': 'label',
    'cv_folds': 3,
    'scoring': 'f1_macro',
    'tfidf': dict(
        ngram_range=(1, 2),
        min_df=1,
        max_df=1.0,
        sublinear_tf=True,
        norm='l2',
    ),
    'class_map': {
        0: 'Non-Toxic',
        1: 'Insults/Flaming',
        2: 'Other Offensive',
        3: 'Hate/Harassment',
        4: 'Threats',
        5: 'Extremism',
    },
}

np.random.seed(CONFIG['seed'])
seed = CONFIG['seed']
tc, lc = CONFIG['text_col'], CONFIG['label_col']

cv = StratifiedKFold(
    n_splits=CONFIG['cv_folds'],
    shuffle=True,
    random_state=seed,
)

print('CONFIG loaded. Text column:', tc)


CONFIG loaded. Text column: message


In [9]:
# load WOT train data
d = CONFIG['data_dir']

wot_train_x = pd.read_parquet(d / 'wot' / 'x_train.parquet')
wot_train_y = wot_train_x[lc]
wot_train_y_binary = (wot_train_y != 0).astype(int)

print('WOT train shape:', wot_train_x.shape)
print('WOT train class distribution:\n', wot_train_y.value_counts())
print('WOT train binary distribution:\n', wot_train_y_binary.value_counts())


WOT train shape: (31401, 2)
WOT train class distribution:
 label
0.0    24942
1.0     4841
2.0     1328
3.0      212
4.0       54
5.0       24
Name: count, dtype: int64
WOT train binary distribution:
 label
0    24942
1     6459
Name: count, dtype: int64


In [10]:
# load DOTA train data
dota_train_x = pd.read_parquet(d / 'dota' / 'x_train.parquet')
dota_train_y = dota_train_x[lc]
dota_train_y_binary = (dota_train_y != 0).astype(int)

print('DOTA train shape:', dota_train_x.shape)
print('DOTA train class distribution:\n', dota_train_y.value_counts())
print('DOTA train binary distribution:\n', dota_train_y_binary.value_counts())


DOTA train shape: (22306, 2)
DOTA train class distribution:
 label
0    15987
1     3518
2     1478
3     1323
Name: count, dtype: int64
DOTA train binary distribution:
 label
0    15987
1     6319
Name: count, dtype: int64


In [11]:
def _fold_series(values, idx):
    return values.iloc[idx] if hasattr(values, 'iloc') else values[idx]


def _safe_confusion_rates(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        'FPR': fp / (fp + tn) if (fp + tn) > 0 else 0,
        'FNR': fn / (fn + tp) if (fn + tp) > 0 else 0,
        'TPR': tp / (tp + fn) if (tp + fn) > 0 else 0,
        'TNR': tn / (tn + fp) if (tn + fp) > 0 else 0,
    }


def _mean_result(dataset_name, model_name, params, fold_metrics, elapsed):
    result = {
        'Dataset': dataset_name,
        'Model': model_name,
        **params,
        'CV Macro F1': round(np.mean([m['CV Macro F1'] for m in fold_metrics]), 4),
        'CV Weighted F1': round(np.mean([m['CV Weighted F1'] for m in fold_metrics]), 4),
        'Accuracy': round(np.mean([m['Accuracy'] for m in fold_metrics]), 4),
        'FPR': round(np.mean([m['FPR'] for m in fold_metrics]), 4),
        'FNR': round(np.mean([m['FNR'] for m in fold_metrics]), 4),
        'TPR': round(np.mean([m['TPR'] for m in fold_metrics]), 4),
        'TNR': round(np.mean([m['TNR'] for m in fold_metrics]), 4),
        'Std Macro F1': round(np.std([m['CV Macro F1'] for m in fold_metrics]), 4),
        'Time (s)': round(elapsed, 1),
    }
    return result


def _prediction_metrics(y_true, y_pred):
    return {
        'CV Macro F1': f1_score(y_true, y_pred, average='macro'),
        'CV Weighted F1': f1_score(y_true, y_pred, average='weighted'),
        'Accuracy': accuracy_score(y_true, y_pred),
        **_safe_confusion_rates(y_true, y_pred),
    }


def evaluate_isolation_forest(X, y, dataset_name, contaminations=None):
    contaminations = contaminations or [0.01, 0.03, 0.05, 0.10, 0.15, 0.20]
    results = []

    for contamination in contaminations:
        t0 = time.time()
        fold_metrics = []

        for train_idx, val_idx in cv.split(X, y):
            X_train_fold = _fold_series(X, train_idx)
            y_train_fold = _fold_series(y, train_idx)
            X_val_fold = _fold_series(X, val_idx)
            y_val_fold = _fold_series(y, val_idx)
            X_train_normal = X_train_fold[y_train_fold == 0]

            pipe = Pipeline([
                ('tfidf', TfidfVectorizer(**CONFIG['tfidf'])),
                ('clf', IsolationForest(
                    random_state=seed,
                    n_jobs=-1,
                    contamination=contamination,
                )),
            ])
            pipe.fit(X_train_normal)
            y_pred = np.where(pipe.predict(X_val_fold) == 1, 0, 1)
            fold_metrics.append(_prediction_metrics(y_val_fold, y_pred))

        results.append(_mean_result(
            dataset_name,
            'Isolation Forest',
            {'Contamination': contamination},
            fold_metrics,
            time.time() - t0,
        ))

    return pd.DataFrame(results).sort_values('CV Macro F1', ascending=False).reset_index(drop=True)


def evaluate_one_class_svm_svd(X, y, dataset_name, nus=None, gammas=None, n_components_list=None):
    nus = nus or [0.01, 0.05, 0.10]
    gammas = gammas or ['scale', 0.01]
    n_components_list = n_components_list or [50, 100]
    results = []

    for n_components in n_components_list:
        for gamma in gammas:
            for nu in nus:
                t0 = time.time()
                fold_metrics = []

                for train_idx, val_idx in cv.split(X, y):
                    X_train_fold = _fold_series(X, train_idx)
                    y_train_fold = _fold_series(y, train_idx)
                    X_val_fold = _fold_series(X, val_idx)
                    y_val_fold = _fold_series(y, val_idx)
                    X_train_normal = X_train_fold[y_train_fold == 0]

                    pipe = Pipeline([
                        ('tfidf', TfidfVectorizer(**CONFIG['tfidf'])),
                        ('svd', TruncatedSVD(n_components=n_components, random_state=seed)),
                        ('scaler', StandardScaler()),
                        ('clf', OneClassSVM(kernel='rbf', gamma=gamma, nu=nu)),
                    ])
                    pipe.fit(X_train_normal)
                    y_pred = np.where(pipe.predict(X_val_fold) == 1, 0, 1)
                    fold_metrics.append(_prediction_metrics(y_val_fold, y_pred))

                results.append(_mean_result(
                    dataset_name,
                    'One-Class SVM + SVD',
                    {'n_components': n_components, 'gamma': gamma, 'nu': nu},
                    fold_metrics,
                    time.time() - t0,
                ))

    return pd.DataFrame(results).sort_values('CV Macro F1', ascending=False).reset_index(drop=True)


def evaluate_centroid_distance(X, y, dataset_name, threshold_percentiles=None):
    threshold_percentiles = threshold_percentiles or [80, 85, 90, 92, 95, 97, 99]
    results = []

    for percentile in threshold_percentiles:
        t0 = time.time()
        fold_metrics = []

        for train_idx, val_idx in cv.split(X, y):
            X_train_fold = _fold_series(X, train_idx)
            y_train_fold = _fold_series(y, train_idx)
            X_val_fold = _fold_series(X, val_idx)
            y_val_fold = _fold_series(y, val_idx)
            X_train_normal = X_train_fold[y_train_fold == 0]

            vectorizer = TfidfVectorizer(**CONFIG['tfidf'])
            X_train_normal_tfidf = vectorizer.fit_transform(X_train_normal)
            X_val_tfidf = vectorizer.transform(X_val_fold)

            normal_centroid = np.asarray(X_train_normal_tfidf.mean(axis=0))
            val_distances = 1 - cosine_similarity(X_val_tfidf, normal_centroid).ravel()
            train_distances = 1 - cosine_similarity(X_train_normal_tfidf, normal_centroid).ravel()
            threshold = np.percentile(train_distances, percentile)

            y_pred = np.where(val_distances > threshold, 1, 0)
            fold_metrics.append(_prediction_metrics(y_val_fold, y_pred))

        results.append(_mean_result(
            dataset_name,
            'Centroid Distance',
            {'Threshold Percentile': percentile},
            fold_metrics,
            time.time() - t0,
        ))

    return pd.DataFrame(results).sort_values('CV Macro F1', ascending=False).reset_index(drop=True)


def run_anomaly_suite(X, y, dataset_name):
    print(f'Running anomaly models for {dataset_name}...')
    model_results = {
        'isolation_forest': evaluate_isolation_forest(X, y, dataset_name),
        'one_class_svm_svd': evaluate_one_class_svm_svd(X, y, dataset_name),
        'centroid_distance': evaluate_centroid_distance(X, y, dataset_name),
    }
    all_results = pd.concat(model_results.values(), ignore_index=True)
    all_results = all_results.sort_values('CV Macro F1', ascending=False).reset_index(drop=True)
    best_model = all_results.head(1).copy()
    return model_results, all_results, best_model


def best_models_by_dataset(results_df):
    return (
        results_df.sort_values('CV Macro F1', ascending=False)
        .groupby('Dataset', as_index=False)
        .head(1)
        .reset_index(drop=True)
    )


In [12]:
# train WOT normal baseline, train DOTA normal baseline, and merged WOT+DOTA normal baseline
anomaly_datasets = {
    'WOT train': (
        wot_train_x[tc].fillna(''),
        wot_train_y_binary.reset_index(drop=True),
    ),
    'DOTA train': (
        dota_train_x[tc].fillna(''),
        dota_train_y_binary.reset_index(drop=True),
    ),
    'Merged WOT+DOTA train': (
        pd.concat([wot_train_x[tc], dota_train_x[tc]], ignore_index=True).fillna(''),
        pd.concat([wot_train_y_binary, dota_train_y_binary], ignore_index=True),
    ),
}

suite_results = {}
all_anomaly_results = []
best_anomaly_models = []

for dataset_name, (X_dataset, y_dataset) in anomaly_datasets.items():
    model_results, dataset_results, dataset_best = run_anomaly_suite(
        X_dataset.reset_index(drop=True),
        y_dataset.reset_index(drop=True),
        dataset_name,
    )
    suite_results[dataset_name] = model_results
    all_anomaly_results.append(dataset_results)
    best_anomaly_models.append(dataset_best)

all_anomaly_results_df = pd.concat(all_anomaly_results, ignore_index=True)
best_anomaly_models_df = pd.concat(best_anomaly_models, ignore_index=True)
best_anomaly_models_df


Running anomaly models for WOT train...
Running anomaly models for DOTA train...
Running anomaly models for Merged WOT+DOTA train...


,Dataset,Model,Contamination,CV Macro F1,CV Weighted F1,Accuracy,FPR,FNR,TPR,TNR,Std Macro F1,Time (s),n_components,gamma,nu,Threshold Percentile
0,WOT train,Centroid Distance,NaN,0.5832,0.7401,0.7537,0.1229,0.7226,0.2774,0.8771,0.0032,0.5,NaN,NaN,NaN,90.0
1,DOTA train,Isolation Forest,0.2,0.5172,0.6194,0.6323,0.2182,0.7457,0.2543,0.7818,0.0073,1.0,NaN,NaN,NaN,NaN
2,Merged WOT+DOTA train,Centroid Distance,NaN,0.5197,0.6600,0.6689,0.1954,0.7656,0.2344,0.8046,0.0034,0.9,NaN,NaN,NaN,80.0


In [13]:
# Full leaderboard across all datasets and all model settings
all_anomaly_results_df.sort_values(
    ['Dataset', 'CV Macro F1'],
    ascending=[True, False],
).reset_index(drop=True)


,Dataset,Model,Contamination,CV Macro F1,CV Weighted F1,Accuracy,FPR,FNR,TPR,TNR,Std Macro F1,Time (s),n_components,gamma,nu,Threshold Percentile
0,DOTA train,Isolation Forest,0.20,0.5172,0.6194,0.6323,0.2182,0.7457,0.2543,0.7818,0.0073,1.0,NaN,NaN,NaN,NaN
1,DOTA train,Isolation Forest,0.15,0.5043,0.6207,0.6497,0.1689,0.8091,0.1909,0.8311,0.0051,1.0,NaN,NaN,NaN,NaN
2,DOTA train,Isolation Forest,0.10,0.4937,0.6236,0.6712,0.1188,0.8601,0.1399,0.8812,0.0046,1.0,NaN,NaN,NaN,NaN
3,DOTA train,One-Class SVM + SVD,NaN,0.4928,0.6132,0.6452,0.1675,0.8288,0.1712,0.8325,0.0066,7.9,100.0,scale,0.1,NaN
4,DOTA train,One-Class SVM + SVD,NaN,0.4928,0.6132,0.6452,0.1675,0.8288,0.1712,0.8325,0.0066,8.3,100.0,0.01,0.1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,WOT train,Isolation Forest,0.20,0.4649,0.6528,0.6552,0.2123,0.8565,0.1435,0.7877,0.0166,1.3,NaN,NaN,NaN,NaN
71,WOT train,Isolation Forest,0.05,0.4616,0.6958,0.7556,0.0592,0.9594,0.0406,0.9408,0.0074,1.4,NaN,NaN,NaN,NaN
72,WOT train,Isolation Forest,0.03,0.4575,0.7004,0.7715,0.0355,0.9738,0.0262,0.9645,0.0061,1.3,NaN,NaN,NaN,NaN
73,WOT train,Isolation Forest,0.01,0.4490,0.7027,0.7861,0.0129,0.9904,0.0096,0.9871,0.0007,1.4,NaN,NaN,NaN,NaN


Among anomaly-detection baselines, centroid distance on the WOT training set achieved the best macro-F1 score of 0.5832. However, its anomaly recall remained low, with a TPR of 0.2774 and an FNR of 0.7226. This indicates that while the method can identify some messages that are textually distant from normal examples, most anomalous messages remain close enough to normal messages in TF-IDF space to avoid detection. Isolation Forest on DOTA and centroid distance on the merged dataset performed worse, suggesting that adding cross-domain data did not improve unsupervised anomaly detection and may have introduced additional distributional variation.